# 【notebook｜N-20】工程环境与工具链
> 对应理论篇：《PY3-工程与环境.md》【文档｜DOC-PY3】
> 配套代码：`00-预备/code/py3_engineering.py`（Tag：`F-pre-py3`）
> CPU 可完整跑通；GPU/依赖相关段落按可用性自动跳过。

## 学习目标
1. 能创建并隔离虚拟环境，正确识别 CPU/CUDA/ROCm 计算后端；
2. 会用 `argparse` + `logging` 组织可配置、可观测的脚本；
3. 会用 `assert`、`pdb`、`pytest` 做调试与测试；
4. 会固定随机种子并从干净内核重跑 notebook，保证可复现。


## 1. 运行环境自检（解释器与后端）


In [ ]:
# 【N-20.c1】
import sys, platform, os
print('python   :', sys.version.split()[0])
print('exe      :', sys.executable)
print('platform :', platform.platform())
print('cwd      :', os.getcwd())

计算后端用 `torch.version.cuda` / `torch.version.hip` 与
`torch.cuda.is_available()` 判定；未安装 `torch` 时下面这段自动跳过，
这正是 PY3 强调的“CPU 路径必须独立可跑”。


In [ ]:
# 【N-20.c2】
try:
    import torch
    print('torch    :', torch.__version__)
    print('cuda ver :', torch.version.cuda)
    print('hip ver  :', torch.version.hip)
    print('cuda可用 :', torch.cuda.is_available())
except ImportError:
    print('torch 未安装：跳过后端检测（CPU 教学路径不受影响）')

## 2. argparse：把配置从源码里挪出来


`--seed/--steps/--log-level` 对应命令行参数；`type=`/`choices=` 在解析阶段
就完成转换与校验（对照 C 的 `getopt`）。


In [ ]:
# 【N-20.c3】
import argparse

def make_parser():
    p = argparse.ArgumentParser(prog='n20-demo', description='PY3 argparse 演示')
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--steps', type=int, default=3)
    p.add_argument('--log-level', default='INFO',
                   choices=['DEBUG', 'INFO', 'WARNING', 'ERROR'])
    return p

args = make_parser().parse_args(['--seed', '1', '--steps', '2'])
print(args)

## 3. logging：分级、格式与通道


In [ ]:
# 【N-20.c4】
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
)
log = logging.getLogger('n20')
log.debug('debug 明细：这一步只在 DEBUG 输出')
log.info('info 汇总：seed=%d steps=%d', args.seed, args.steps)
log.warning('warning 提示')

## 4. 可复现性：随机种子与断言


同一 `seed` 必须得到同一结果；`assert` 用作开发期契约
（注意 `python -O` 会移除 assert，不能校验外部输入）。


In [ ]:
# 【N-20.c5】
import random

def draw(seed, n=5):
    random.seed(seed)
    return [random.random() for _ in range(n)]

assert draw(42) == draw(42), '同种子必须可复现'
assert draw(0) != draw(1), '不同种子（高概率）不同'
print('draw(42) =', draw(42))

调试时在可疑处插入 `breakpoint()`（对照 gdb 断点），可交互执行
`n`/`s`/`c`/`p expr`/`q`；异常后可用 `import pdb; pdb.pm()` 做事后调试。
下面用 try/except 演示断言触发的失败路径。


In [ ]:
# 【N-20.c6】
def safe_mean(values):
    assert len(values) > 0, 'values 不能为空'
    return sum(values) / len(values)

print('mean =', safe_mean([1.0, 2.0, 3.0]))
try:
    safe_mean([])
except AssertionError as e:
    print('捕获到断言失败：', e)

## 5. pytest：把上面的检查固化成测试


`pytest` 递归收集 `test_*.py` 的 `test_*` 函数；`@pytest.fixture` 提供
前置资源，`@pytest.mark.parametrize` 生成多条用例。这里直接在 cell 内
调用测试函数，等价于最小测试运行器。


In [ ]:
# 【N-20.c7】
def test_seed_determinism():
    assert draw(7) == draw(7)

def test_different_seeds_differ():
    assert draw(0) != draw(1)

for name, fn in sorted(globals().items()):
    if name.startswith('test_') and callable(fn):
        fn()
        print('PASS', name)

## 6. git 快照与工作树（只读演示）


`git add <自己的文件>` -> `git commit`；`git worktree add` 可在一个仓库上
开出多个独立工作目录（本团队用 `tm/<team>/<name>` 分支，只提交自己的文件）。


In [ ]:
# 【N-20.c8】
import subprocess, pathlib

def git(*argv, cwd=None):
    r = subprocess.run(['git', *argv], cwd=cwd, capture_output=True, text=True)
    return r.returncode, r.stdout.strip()

print('git version:', git('--version')[1])
root = pathlib.Path('.').resolve()
rc, out = git('rev-parse', '--show-toplevel')
print('toplevel  :', out if rc == 0 else '(非 git 仓库，跳过)')

## 7. 从干净内核重跑（可复现交付）

命令行执行（把 `!` 换成 shell）：

```bash
jupyter nbconvert --to notebook --execute --inplace N20_工程环境与工具链.ipynb
```

它等价于界面上的 `Restart Kernel and Run All`，确保 cell 执行顺序
与结果可复现（见理论篇【注 PY3.5.1】）。


## 思考题

1. 为什么 `sudo pip install` 会破坏虚拟环境隔离？
2. `python -O` 下 `assert` 会消失，业务校验应改用什么？
3. 续跑实验要记录哪三类信息才称得上“可复现”？

## 改动实验（任选 1–2 个）

1. 给 `make_parser()` 增加 `--repeat`，并让 `logging.debug` 打印每步明细。
2. 把 `draw()` 改成可选使用 `numpy.random.default_rng(seed)` 并比较结果。
3. 用 `git worktree add` 为某个实验单开一个目录，提交后删除。
